# Iterative Reference-guided Slice-wise XY Translation Registration with Elastix

이 노트북은 각 `stacks` 폴더의 `Reference_stack.tif`에서 z-slice별 drift 보정 transform을 Elastix로 추정한 뒤, 같은 transform을 동일 z 위치의 `DAPI_stack.tif`, `Reference_stack.tif`, `Target_stack.tif`에 함께 적용합니다.

- `itk-elastix`: `Reference_stack.tif`의 n번째 z slice를 보정된 n-1번째 reference slice에 맞추는 iterative x-y translation registration 수행
- `DAPI_stack.tif` / `Target_stack.tif`: 별도로 registration하지 않고 reference에서 추정한 동일 transform만 적용
- 목적: 같은 샘플의 z-stack drift를 순차적으로 보정하면서 DAPI-reference-target 채널 간 상대 거리/offset을 유지
- `SimpleITK`: TIFF stack을 이미지 객체로 다루고 최종 transform을 원본 해상도 slice에 resampling하는 보조 용도
- metric: adjacent reference slice 간 보정 전후의 `mean_squares`, `mutual_information`, `normalized_mutual_information`, `pearson_r`

## 0. Optional Install

필요한 패키지가 없는 환경에서만 주석을 해제해 설치합니다.


In [ ]:
# 필요 시 먼저 실행하세요.
# 네트워크가 가능한 환경에서 아래 셀의 주석을 해제하면 됩니다.

# %pip install -q SimpleITK itk-elastix tifffile pandas matplotlib


## 1. Imports and Runtime Configuration

라이브러리, 의존성 체크, registration 기본 파라미터를 한 곳에서 설정합니다.


In [ ]:
from __future__ import annotations

from pathlib import Path
from io import BytesIO
import base64
import importlib.util
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from PIL import Image, ImageDraw
from IPython.display import HTML, display

HAS_SIMPLEITK = importlib.util.find_spec("SimpleITK") is not None
HAS_ITK = importlib.util.find_spec("itk") is not None

if not HAS_SIMPLEITK:
    raise ImportError("SimpleITK가 설치되어 있지 않습니다. 위 설치 셀을 먼저 실행하세요.")
if not HAS_ITK:
    raise ImportError("itk / itk-elastix가 설치되어 있지 않습니다. 위 설치 셀을 먼저 실행하세요.")

import SimpleITK as sitk

ROOT = Path.cwd()
CHANNEL_TIFF_NAMES = {
    "DAPI": "DAPI_stack.tif",
    "Reference": "Reference_stack.tif",
    "Target": "Target_stack.tif",
}
CHANNEL_LABELS = tuple(CHANNEL_TIFF_NAMES.keys())
REFERENCE_LABEL = "Reference"
TARGET_LABEL = "Target"
VOXEL_SPACING_ZYX = (1.0, 1.0, 1.0)
CLIP_PERCENTILES = (1.0, 99.8)
GAUSSIAN_SIGMA = 1.0
MIN_SLICE_FOREGROUND_PIXELS = 256
QUICK_TEST_MODE = True
REGISTRATION_XY_DOWNSAMPLE_FACTOR = 4
ELASTIX_MAX_ITERATIONS = 96
ELASTIX_METRIC = "AdvancedMattesMutualInformation"
JUMP_THRESHOLD_VOXELS = 5.0
FATAL_JUMP_THRESHOLD_VOXELS = 10.0

print({
    "root": str(ROOT),
    "has_simpleitk": HAS_SIMPLEITK,
    "has_itk_elastix": HAS_ITK,
    "channels": CHANNEL_TIFF_NAMES,
    "reference_channel": REFERENCE_LABEL,
    "registration_engine": "itk-elastix",
    "registration_strategy": "iterative previous-slice reference guided, apply transform to all channels",
})

## 2. Output Options

파일 저장 여부와 저장 파일명을 제어합니다. 화면 확인만 할 때는 저장 옵션을 `False`로 둘 수 있습니다.


In [ ]:
# 저장 옵션은 이 셀에서만 조절합니다.
# 기본값 False: notebook 화면에만 표시하고 파일로 저장하지 않습니다.
SAVE_REGISTERED_TIFF_STACKS = True
SAVE_METRICS_SUMMARY_CSV = False
SAVE_TRANSFORM_PARAMETERS_CSV = False
SAVE_TRANSFORM_SHIFT_PLOTS = False
SAVE_OVERLAY_GIFS = False
SAVE_METADATA_JSON = True

REGISTERED_STACKS_DIR_NAME = "registered_stacks"
METRICS_OUTPUT_CSV = ROOT / "registration_metrics_summary.csv"
TRANSFORM_PARAMETERS_CSV_NAME = "transform_parameters.csv"
TRANSFORM_SHIFT_PLOT_NAME = "transform_shift.png"
BEFORE_OVERLAY_GIF_NAME = "before_registration_overlay_ref_green_target_red.gif"
AFTER_OVERLAY_GIF_NAME = "after_registration_overlay_ref_green_target_red.gif"

print({
    "save_registered_tiff_stacks": SAVE_REGISTERED_TIFF_STACKS,
    "save_metrics_summary_csv": SAVE_METRICS_SUMMARY_CSV,
    "save_transform_parameters_csv": SAVE_TRANSFORM_PARAMETERS_CSV,
    "save_transform_shift_plots": SAVE_TRANSFORM_SHIFT_PLOTS,
    "save_overlay_gifs": SAVE_OVERLAY_GIFS,
    "save_metadata_json": SAVE_METADATA_JSON,
    "registered_stacks_dir_name": REGISTERED_STACKS_DIR_NAME,
    "metrics_output_csv": str(METRICS_OUTPUT_CSV),
})

## 3. Stack Discovery and Preprocessing

`stacks` 폴더를 찾고 TIFF stack을 SimpleITK 이미지로 변환한 뒤 registration용 intensity preprocessing을 수행합니다.


In [ ]:
def discover_stack_sets(root: Path, channel_tiff_names: dict[str, str] = CHANNEL_TIFF_NAMES) -> pd.DataFrame:
    rows = []
    for stacks_dir in sorted(root.glob("**/stacks")):
        paths = {label: stacks_dir / filename for label, filename in channel_tiff_names.items()}
        if all(path.exists() for path in paths.values()):
            sample_id = str(stacks_dir.parent.relative_to(root))
            row = {
                "sample_id": sample_id,
                "stacks_dir": str(stacks_dir),
                "series_dir": str(stacks_dir.parent),
            }
            for label, path in paths.items():
                row[f"{label.lower()}_path"] = str(path)
            rows.append(row)
    return pd.DataFrame(rows)


# Backward-compatible alias for older cells/scripts that still call this name.
def discover_stack_pairs(root: Path, fixed_name: str | None = None, moving_name: str | None = None) -> pd.DataFrame:
    del fixed_name, moving_name
    return discover_stack_sets(root)


def read_tiff_as_sitk(path: Path, spacing_zyx=VOXEL_SPACING_ZYX) -> sitk.Image:
    arr = tifffile.imread(path).astype(np.float32)
    img = sitk.GetImageFromArray(arr)
    img.SetSpacing(tuple(spacing_zyx[::-1]))
    img.SetOrigin((0.0, 0.0, 0.0))
    return img


def load_channel_images(row, channel_labels: tuple[str, ...] = CHANNEL_LABELS) -> dict[str, sitk.Image]:
    return {
        label: read_tiff_as_sitk(Path(getattr(row, f"{label.lower()}_path")))
        for label in channel_labels
    }


def preprocess_image(image: sitk.Image, clip_percentiles=CLIP_PERCENTILES, sigma=GAUSSIAN_SIGMA) -> sitk.Image:
    arr = sitk.GetArrayFromImage(image).astype(np.float32)
    valid = arr[arr > 0]
    base = valid if valid.size else arr.reshape(-1)
    lo, hi = np.percentile(base, clip_percentiles)
    hi = max(hi, lo + 1e-6)
    arr = np.clip(arr, lo, hi)
    arr = (arr - lo) / (hi - lo)
    out = sitk.GetImageFromArray(arr)
    out.CopyInformation(image)
    if sigma and sigma > 0:
        out = sitk.DiscreteGaussian(out, variance=float(sigma) ** 2)
    return sitk.Cast(out, sitk.sitkFloat32)

## 4. Registration Quality Metrics

보정 전후 adjacent reference slice 유사도를 mean-squares, MI, NMI, Pearson r로 계산합니다.


In [ ]:
def build_joint_mask(fixed_arr: np.ndarray, moving_arr: np.ndarray) -> np.ndarray:
    fixed_pos = fixed_arr[fixed_arr > 0]
    moving_pos = moving_arr[moving_arr > 0]
    fixed_thr = np.percentile(fixed_pos, 20) if fixed_pos.size else 0.0
    moving_thr = np.percentile(moving_pos, 20) if moving_pos.size else 0.0
    mask = (fixed_arr > fixed_thr) | (moving_arr > moving_thr)
    if mask.sum() < 1024:
        mask = np.ones_like(mask, dtype=bool)
    return mask


def compute_mutual_information(x: np.ndarray, y: np.ndarray, bins: int = 64) -> float:
    hist_2d, _, _ = np.histogram2d(x, y, bins=bins)
    pxy = hist_2d / np.maximum(hist_2d.sum(), 1.0)
    px = pxy.sum(axis=1)
    py = pxy.sum(axis=0)
    px_py = np.outer(px, py)
    nz = pxy > 0
    return float(np.sum(pxy[nz] * np.log((pxy[nz] + 1e-12) / (px_py[nz] + 1e-12))))


def compute_normalized_mutual_information(x: np.ndarray, y: np.ndarray, bins: int = 64) -> float:
    hist_2d, _, _ = np.histogram2d(x, y, bins=bins)
    pxy = hist_2d / np.maximum(hist_2d.sum(), 1.0)
    px = pxy.sum(axis=1)
    py = pxy.sum(axis=0)
    hx = -np.sum(px[px > 0] * np.log(px[px > 0] + 1e-12))
    hy = -np.sum(py[py > 0] * np.log(py[py > 0] + 1e-12))
    mi = compute_mutual_information(x, y, bins=bins)
    return float((hx + hy) / np.maximum(mi, 1e-12))


def evaluate_pair_metrics(fixed_image: sitk.Image, moving_image: sitk.Image) -> dict:
    fixed_arr = sitk.GetArrayFromImage(fixed_image).astype(np.float32)
    moving_arr = sitk.GetArrayFromImage(moving_image).astype(np.float32)
    mask = build_joint_mask(fixed_arr, moving_arr)
    x = fixed_arr[mask].ravel()
    y = moving_arr[mask].ravel()
    if x.size == 0 or y.size == 0:
        raise ValueError("Metric 계산에 사용할 foreground voxel이 없습니다.")
    mean_squares = float(np.mean((x - y) ** 2))
    if x.std() < 1e-12 or y.std() < 1e-12:
        pearson_r = float("nan")
    else:
        pearson_r = float(np.corrcoef(x, y)[0, 1])
    return {
        "mean_squares": mean_squares,
        "mutual_information": compute_mutual_information(x, y),
        "normalized_mutual_information": compute_normalized_mutual_information(x, y),
        "pearson_r": pearson_r,
        "n_voxels": int(x.size),
    }


def metric_delta(before: dict, after: dict) -> dict:
    return {k: float(after[k] - before[k]) for k in before.keys() if k != "n_voxels"}


def evaluate_adjacent_stack_metrics(image: sitk.Image) -> dict:
    arr = sitk.GetArrayFromImage(image).astype(np.float32)
    if arr.shape[0] < 2:
        raise ValueError("Adjacent metric 계산에는 최소 2개 z slice가 필요합니다.")
    previous_stack = sitk.GetImageFromArray(arr[:-1])
    current_stack = sitk.GetImageFromArray(arr[1:])
    return evaluate_pair_metrics(previous_stack, current_stack)


## 5. SimpleITK Slice Utilities

3D stack과 2D slice 사이의 변환, downsampling, signal check, transform 요약에 쓰는 보조 함수입니다.


In [ ]:
def sitk_image_to_array(image: sitk.Image) -> np.ndarray:
    return sitk.GetArrayFromImage(image).astype(np.float32)


def make_2d_sitk_image(slice_arr: np.ndarray, reference_3d: sitk.Image, z_index: int) -> sitk.Image:
    image_2d = sitk.GetImageFromArray(slice_arr.astype(np.float32))
    spacing = reference_3d.GetSpacing()
    origin = reference_3d.GetOrigin()
    image_2d.SetSpacing((spacing[0], spacing[1]))
    image_2d.SetOrigin((origin[0], origin[1]))
    return image_2d


def compose_stack_from_slices(slices: list[np.ndarray], reference_3d: sitk.Image) -> sitk.Image:
    stack = sitk.GetImageFromArray(np.stack(slices, axis=0).astype(np.float32))
    stack.CopyInformation(reference_3d)
    return sitk.Cast(stack, sitk.sitkFloat32)


def downsample_image_for_registration(image: sitk.Image, factor: int = REGISTRATION_XY_DOWNSAMPLE_FACTOR) -> sitk.Image:
    if factor is None or factor <= 1:
        return image
    size = list(image.GetSize())
    spacing = list(image.GetSpacing())
    new_size = [max(16, int(round(size[0] / factor))), max(16, int(round(size[1] / factor)))]
    new_spacing = [spacing[0] * size[0] / new_size[0], spacing[1] * size[1] / new_size[1]]
    return sitk.Resample(
        image,
        new_size,
        sitk.Transform(),
        sitk.sitkLinear,
        image.GetOrigin(),
        new_spacing,
        image.GetDirection(),
        0.0,
        image.GetPixelID(),
    )


def slice_has_signal(fixed_slice: np.ndarray, moving_slice: np.ndarray, min_pixels: int = MIN_SLICE_FOREGROUND_PIXELS) -> bool:
    return int(np.count_nonzero(fixed_slice > 0) + np.count_nonzero(moving_slice > 0)) >= min_pixels


def summarize_slice_transforms(slice_transforms: list[list[float]]) -> str:
    if not slice_transforms:
        return "no successful slice transforms"
    arr = np.asarray(slice_transforms, dtype=np.float32)
    mean_vals = np.round(arr.mean(axis=0), 5).tolist()
    std_vals = np.round(arr.std(axis=0), 5).tolist()
    return f"n={len(slice_transforms)}, mean={mean_vals}, std={std_vals}"


## 6. GIF and HTML Visualization Helpers

registration 전후 stack, overlay, 처리 단계 비교를 notebook 안에서 GIF로 표시하는 함수입니다.


In [ ]:
def normalize_slice_for_gif(slice_arr: np.ndarray, p_low: float = 1.0, p_high: float = 99.8) -> np.ndarray:
    slice_arr = np.asarray(slice_arr, dtype=np.float32)
    if slice_arr.size == 0 or np.allclose(slice_arr, slice_arr.flat[0]):
        return np.zeros_like(slice_arr, dtype=np.float32)
    lo, hi = np.percentile(slice_arr, (p_low, p_high))
    hi = max(hi, lo + 1e-6)
    return np.clip((slice_arr - lo) / (hi - lo), 0.0, 1.0).astype(np.float32)


def add_slice_label(frame: Image.Image, z_index: int, label_prefix: str = "z") -> Image.Image:
    frame = frame.convert("RGB")
    draw = ImageDraw.Draw(frame)
    label = f"{label_prefix}={z_index}"
    bbox = draw.textbbox((0, 0), label)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]
    pad = 4
    x = max(0, frame.width - text_w - 2 * pad)
    y = max(0, frame.height - text_h - 2 * pad)
    draw.rectangle((x, y, frame.width, frame.height), fill=(0, 0, 0))
    draw.text((x + pad, y + pad), label, fill=(255, 255, 255))
    return frame


def encode_pil_frames_to_gif(frames: list[Image.Image], duration_ms: int = 120) -> str:
    if not frames:
        return ""
    buffer = BytesIO()
    frames[0].save(
        buffer,
        format="GIF",
        save_all=True,
        append_images=frames[1:],
        duration=duration_ms,
        loop=0,
        optimize=False,
    )
    return base64.b64encode(buffer.getvalue()).decode("ascii")


def build_grayscale_gif(stack: np.ndarray, duration_ms: int = 120, show_slice_label: bool = True) -> str:
    frames = []
    for z in range(stack.shape[0]):
        frame = normalize_slice_for_gif(stack[z])
        frame_uint8 = np.round(frame * 255.0).astype(np.uint8)
        pil_frame = Image.fromarray(frame_uint8, mode="L")
        if show_slice_label:
            pil_frame = add_slice_label(pil_frame, z)
        frames.append(pil_frame)
    return encode_pil_frames_to_gif(frames, duration_ms=duration_ms)


def save_grayscale_gif(image: sitk.Image, output_path: Path, duration_ms: int = 120) -> Path:
    stack = sitk_image_to_array(image)
    frames = []
    for z in range(stack.shape[0]):
        frame = normalize_slice_for_gif(stack[z])
        frame_uint8 = np.round(frame * 255.0).astype(np.uint8)
        frames.append(Image.fromarray(frame_uint8, mode="L"))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frames[0].save(
        output_path,
        format="GIF",
        save_all=True,
        append_images=frames[1:],
        duration=duration_ms,
        loop=0,
        optimize=False,
    )
    return output_path


def build_overlay_gif(fixed_stack: np.ndarray, moving_stack: np.ndarray, duration_ms: int = 120, show_slice_label: bool = True) -> str:
    frames = []
    for z in range(fixed_stack.shape[0]):
        fixed_norm = normalize_slice_for_gif(fixed_stack[z])
        moving_norm = normalize_slice_for_gif(moving_stack[z])
        rgb = np.zeros((*fixed_norm.shape, 3), dtype=np.uint8)
        rgb[..., 0] = np.round(moving_norm * 255.0).astype(np.uint8)
        rgb[..., 1] = np.round(fixed_norm * 255.0).astype(np.uint8)
        pil_frame = Image.fromarray(rgb, mode="RGB")
        if show_slice_label:
            pil_frame = add_slice_label(pil_frame, z)
        frames.append(pil_frame)
    return encode_pil_frames_to_gif(frames, duration_ms=duration_ms)


def make_registration_gif_comparison_html(sample_id: str, method: str, fixed_image: sitk.Image, moving_image: sitk.Image, registered_image: sitk.Image, duration_ms: int = 120, display_width: int = 240) -> str:
    fixed_stack = sitk_image_to_array(fixed_image)
    moving_stack = sitk_image_to_array(moving_image)
    registered_stack = sitk_image_to_array(registered_image)

    if fixed_stack.shape != moving_stack.shape or fixed_stack.shape != registered_stack.shape:
        raise ValueError("GIF 비교를 위해서는 fixed, moving, registered stack shape이 모두 같아야 합니다.")

    fixed_gif = build_grayscale_gif(fixed_stack, duration_ms=duration_ms)
    moving_gif = build_grayscale_gif(moving_stack, duration_ms=duration_ms)
    registered_gif = build_grayscale_gif(registered_stack, duration_ms=duration_ms)
    before_overlay_gif = build_overlay_gif(fixed_stack, moving_stack, duration_ms=duration_ms)
    after_overlay_gif = build_overlay_gif(fixed_stack, registered_stack, duration_ms=duration_ms)

    tile_style = f"width:{display_width}px;height:auto;display:block;background:black;"
    overlay_style = f"width:{display_width * 2}px;max-width:100%;height:auto;display:block;background:black;"
    return f"""
<div style='background:#050505;padding:14px;border-radius:10px;margin:10px 0 22px 0;'>
  <div style='color:white;font-size:16px;font-weight:700;margin-bottom:10px;'>{sample_id} | {method}</div>
  <div style='display:grid;grid-template-columns:repeat(3, max-content);gap:18px;align-items:start;justify-content:start;'>
    <div><div style='color:white;text-align:center;margin-bottom:6px;'>Fixed</div><img src='data:image/gif;base64,{fixed_gif}' style='{tile_style}' /></div>
    <div><div style='color:white;text-align:center;margin-bottom:6px;'>Before Moving</div><img src='data:image/gif;base64,{moving_gif}' style='{tile_style}' /></div>
    <div><div style='color:white;text-align:center;margin-bottom:6px;'>After Registered</div><img src='data:image/gif;base64,{registered_gif}' style='{tile_style}' /></div>
    <div style='grid-column:1 / span 3;display:grid;grid-template-columns:repeat(2, max-content);gap:18px;'>
      <div><div style='color:white;text-align:center;margin-bottom:6px;'>Before Overlay (Ref=Green, Moving=Red)</div><img src='data:image/gif;base64,{before_overlay_gif}' style='{overlay_style}' /></div>
      <div><div style='color:white;text-align:center;margin-bottom:6px;'>After Overlay (Ref=Green, Registered=Red)</div><img src='data:image/gif;base64,{after_overlay_gif}' style='{overlay_style}' /></div>
    </div>
  </div>
</div>
"""


def show_registration_gif_comparison(sample_id: str, method: str, fixed_image: sitk.Image, moving_image: sitk.Image, registered_image: sitk.Image, duration_ms: int = 120, display_width: int = 240) -> None:
    html = make_registration_gif_comparison_html(
        sample_id=sample_id,
        method=method,
        fixed_image=fixed_image,
        moving_image=moving_image,
        registered_image=registered_image,
        duration_ms=duration_ms,
        display_width=display_width,
    )
    display(HTML(html))




def make_before_after_overlay_gif_html(sample_id: str, method: str, reference_raw_image: sitk.Image, target_raw_image: sitk.Image, registered_reference_image: sitk.Image, registered_target_image: sitk.Image, duration_ms: int = 120, display_width: int = 320) -> str:
    reference_raw_stack = sitk_image_to_array(reference_raw_image)
    target_raw_stack = sitk_image_to_array(target_raw_image)
    registered_reference_stack = sitk_image_to_array(registered_reference_image)
    registered_target_stack = sitk_image_to_array(registered_target_image)

    shapes = {
        reference_raw_stack.shape,
        target_raw_stack.shape,
        registered_reference_stack.shape,
        registered_target_stack.shape,
    }
    if len(shapes) != 1:
        raise ValueError("Before/after overlay GIF 비교를 위해서는 모든 stack shape이 같아야 합니다.")

    before_overlay_gif = build_overlay_gif(reference_raw_stack, target_raw_stack, duration_ms=duration_ms)
    after_overlay_gif = build_overlay_gif(registered_reference_stack, registered_target_stack, duration_ms=duration_ms)
    tile_style = f"width:{display_width}px;height:auto;display:block;background:black;"
    return f"""
<div style='background:#050505;padding:14px;border-radius:10px;margin:10px 0 22px 0;'>
  <div style='color:white;font-size:16px;font-weight:700;margin-bottom:10px;'>{sample_id} | {method} | Reference=Green, Target=Red</div>
  <div style='display:grid;grid-template-columns:repeat(2, max-content);gap:20px;align-items:start;justify-content:start;'>
    <div><div style='color:white;text-align:center;margin-bottom:6px;'>Before registration: raw stacks</div><img src='data:image/gif;base64,{before_overlay_gif}' style='{tile_style}' /></div>
    <div><div style='color:white;text-align:center;margin-bottom:6px;'>After registration: corrected stacks</div><img src='data:image/gif;base64,{after_overlay_gif}' style='{tile_style}' /></div>
  </div>
</div>
"""


def show_before_after_overlay_gif_comparison(sample_id: str, method: str, reference_raw_image: sitk.Image, target_raw_image: sitk.Image, registered_reference_image: sitk.Image, registered_target_image: sitk.Image, duration_ms: int = 120, display_width: int = 320) -> None:
    display(HTML(make_before_after_overlay_gif_html(
        sample_id=sample_id,
        method=method,
        reference_raw_image=reference_raw_image,
        target_raw_image=target_raw_image,
        registered_reference_image=registered_reference_image,
        registered_target_image=registered_target_image,
        duration_ms=duration_ms,
        display_width=display_width,
    )))

def make_processing_stage_gif_html(sample_id: str, method: str, raw_image: sitk.Image, preprocessed_image: sitk.Image, registered_image: sitk.Image, reference_raw_image: sitk.Image, reference_preprocessed_image: sitk.Image, registered_reference_image: sitk.Image, duration_ms: int = 120, display_width: int = 240) -> str:
    raw_stack = sitk_image_to_array(raw_image)
    preprocessed_stack = sitk_image_to_array(preprocessed_image)
    registered_stack = sitk_image_to_array(registered_image)
    reference_raw_stack = sitk_image_to_array(reference_raw_image)
    reference_preprocessed_stack = sitk_image_to_array(reference_preprocessed_image)
    registered_reference_stack = sitk_image_to_array(registered_reference_image)

    shapes = {
        raw_stack.shape,
        preprocessed_stack.shape,
        registered_stack.shape,
        reference_raw_stack.shape,
        reference_preprocessed_stack.shape,
        registered_reference_stack.shape,
    }
    if len(shapes) != 1:
        raise ValueError("GIF 비교를 위해서는 reference, target, registered stack shape이 모두 같아야 합니다.")

    raw_gif = build_grayscale_gif(raw_stack, duration_ms=duration_ms)
    preprocessed_gif = build_grayscale_gif(preprocessed_stack, duration_ms=duration_ms)
    registered_gif = build_grayscale_gif(registered_stack, duration_ms=duration_ms)
    reference_raw_gif = build_grayscale_gif(reference_raw_stack, duration_ms=duration_ms)
    registered_reference_gif = build_grayscale_gif(registered_reference_stack, duration_ms=duration_ms)
    raw_overlay_gif = build_overlay_gif(reference_raw_stack, raw_stack, duration_ms=duration_ms)
    preprocessed_overlay_gif = build_overlay_gif(reference_preprocessed_stack, preprocessed_stack, duration_ms=duration_ms)
    registered_overlay_gif = build_overlay_gif(registered_reference_stack, registered_stack, duration_ms=duration_ms)

    tile_style = f"width:{display_width}px;height:auto;display:block;background:black;"
    overlay_style = f"width:{display_width}px;height:auto;display:block;background:black;"
    return f"""
<div style='background:#050505;padding:14px;border-radius:10px;margin:10px 0 22px 0;'>
  <div style='color:white;font-size:16px;font-weight:700;margin-bottom:10px;'>{sample_id} | {method} | Ref=Green, Target=Red</div>
  <div style='display:grid;grid-template-columns:repeat(3, max-content);gap:18px;align-items:start;justify-content:start;'>
    <div><div style='color:white;text-align:center;margin-bottom:6px;'>Raw Reference_stack</div><img src='data:image/gif;base64,{reference_raw_gif}' style='{tile_style}' /></div>
    <div><div style='color:white;text-align:center;margin-bottom:6px;'>Registered Reference_stack</div><img src='data:image/gif;base64,{registered_reference_gif}' style='{tile_style}' /></div>
    <div><div style='color:white;text-align:center;margin-bottom:6px;'>Raw Target_stack</div><img src='data:image/gif;base64,{raw_gif}' style='{tile_style}' /></div>
    <div><div style='color:white;text-align:center;margin-bottom:6px;'>Preprocessed Target_stack</div><img src='data:image/gif;base64,{preprocessed_gif}' style='{tile_style}' /></div>
    <div><div style='color:white;text-align:center;margin-bottom:6px;'>Registered Target_stack</div><img src='data:image/gif;base64,{registered_gif}' style='{tile_style}' /></div>
    <div><div style='color:white;text-align:center;margin-bottom:6px;'>Raw Overlay</div><img src='data:image/gif;base64,{raw_overlay_gif}' style='{overlay_style}' /></div>
    <div><div style='color:white;text-align:center;margin-bottom:6px;'>Preprocessed Overlay</div><img src='data:image/gif;base64,{preprocessed_overlay_gif}' style='{overlay_style}' /></div>
    <div><div style='color:white;text-align:center;margin-bottom:6px;'>Registered Overlay</div><img src='data:image/gif;base64,{registered_overlay_gif}' style='{overlay_style}' /></div>
  </div>
</div>
"""


def show_processing_stage_gif_comparison(sample_id: str, method: str, raw_image: sitk.Image, preprocessed_image: sitk.Image, registered_image: sitk.Image, reference_raw_image: sitk.Image, reference_preprocessed_image: sitk.Image, registered_reference_image: sitk.Image, duration_ms: int = 120, display_width: int = 240) -> None:
    html = make_processing_stage_gif_html(
        sample_id=sample_id,
        method=method,
        raw_image=raw_image,
        preprocessed_image=preprocessed_image,
        registered_image=registered_image,
        reference_raw_image=reference_raw_image,
        reference_preprocessed_image=reference_preprocessed_image,
        registered_reference_image=registered_reference_image,
        duration_ms=duration_ms,
        display_width=display_width,
    )
    display(HTML(html))


## 7. Output Writers

보정 결과 TIFF stack, overlay GIF, stage GIF, transform plot 관련 파일을 저장하는 함수입니다.


In [ ]:
def save_overlay_gif(fixed_image: sitk.Image, moving_image: sitk.Image, output_path: Path, duration_ms: int = 120) -> Path:
    fixed_stack = sitk_image_to_array(fixed_image)
    moving_stack = sitk_image_to_array(moving_image)
    if fixed_stack.shape != moving_stack.shape:
        raise ValueError("Overlay GIF 저장을 위해서는 fixed, moving stack shape이 같아야 합니다.")
    frames = []
    for z in range(fixed_stack.shape[0]):
        fixed_norm = normalize_slice_for_gif(fixed_stack[z])
        moving_norm = normalize_slice_for_gif(moving_stack[z])
        rgb = np.zeros((*fixed_norm.shape, 3), dtype=np.uint8)
        rgb[..., 0] = np.round(moving_norm * 255.0).astype(np.uint8)
        rgb[..., 1] = np.round(fixed_norm * 255.0).astype(np.uint8)
        frames.append(Image.fromarray(rgb, mode="RGB"))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    frames[0].save(
        output_path,
        format="GIF",
        save_all=True,
        append_images=frames[1:],
        duration=duration_ms,
        loop=0,
        optimize=False,
    )
    return output_path


def save_processing_stage_gifs(sample_id: str, method: str, raw_image: sitk.Image, preprocessed_image: sitk.Image, registered_image: sitk.Image, reference_raw_image: sitk.Image, reference_preprocessed_image: sitk.Image, registered_reference_image: sitk.Image, output_dir: Path, duration_ms: int = 120) -> dict:
    safe_sample_id = sample_id.replace("/", "__").replace(" ", "_")
    safe_method = method.replace(" ", "_")
    paths = {
        "raw_target": output_dir / f"{safe_sample_id}__{safe_method}__raw_target.gif",
        "preprocessed_target": output_dir / f"{safe_sample_id}__{safe_method}__preprocessed_target.gif",
        "registered_target": output_dir / f"{safe_sample_id}__{safe_method}__registered_target.gif",
        "raw_reference": output_dir / f"{safe_sample_id}__{safe_method}__raw_reference.gif",
        "preprocessed_reference": output_dir / f"{safe_sample_id}__{safe_method}__preprocessed_reference.gif",
        "registered_reference": output_dir / f"{safe_sample_id}__{safe_method}__registered_reference.gif",
        "raw_overlay": output_dir / f"{safe_sample_id}__{safe_method}__raw_overlay_ref_green_target_red.gif",
        "preprocessed_overlay": output_dir / f"{safe_sample_id}__{safe_method}__preprocessed_overlay_ref_green_target_red.gif",
        "registered_overlay": output_dir / f"{safe_sample_id}__{safe_method}__registered_overlay_ref_green_target_red.gif",
    }
    save_grayscale_gif(raw_image, paths["raw_target"], duration_ms=duration_ms)
    save_grayscale_gif(preprocessed_image, paths["preprocessed_target"], duration_ms=duration_ms)
    save_grayscale_gif(registered_image, paths["registered_target"], duration_ms=duration_ms)
    save_grayscale_gif(reference_raw_image, paths["raw_reference"], duration_ms=duration_ms)
    save_grayscale_gif(reference_preprocessed_image, paths["preprocessed_reference"], duration_ms=duration_ms)
    save_grayscale_gif(registered_reference_image, paths["registered_reference"], duration_ms=duration_ms)
    save_overlay_gif(reference_raw_image, raw_image, paths["raw_overlay"], duration_ms=duration_ms)
    save_overlay_gif(reference_preprocessed_image, preprocessed_image, paths["preprocessed_overlay"], duration_ms=duration_ms)
    save_overlay_gif(registered_reference_image, registered_image, paths["registered_overlay"], duration_ms=duration_ms)
    return paths


def cast_array_like_tiff_dtype(arr: np.ndarray, reference_tiff_path: Path) -> np.ndarray:
    with tifffile.TiffFile(reference_tiff_path) as tif:
        output_dtype = np.dtype(tif.series[0].dtype)

    if np.issubdtype(output_dtype, np.integer):
        limits = np.iinfo(output_dtype)
        arr = np.clip(np.rint(arr), limits.min, limits.max)
    return arr.astype(output_dtype, copy=False)


def write_tiff_stack(output_path: Path, arr: np.ndarray) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    tifffile.imwrite(
        output_path,
        arr,
        photometric="minisblack",
        bigtiff=arr.nbytes > 4_000_000_000,
    )
    return output_path


def build_transform_provenance_records(
    transform_parameters: list[list[float]],
    sample_id: str,
    method: str,
    registered_stacks_dir: Path,
) -> list[dict]:
    """Return the same per-slice transform fields used by the optional CSV."""
    records = []
    for z, values in enumerate(transform_parameters):
        if len(values) < 3:
            continue
        x_shift = float(values[1])
        y_shift = float(values[2])
        records.append({
            "sample_id": sample_id,
            "method": method,
            "registered_stacks_dir": str(registered_stacks_dir),
            "z": int(z),
            "x_shift": x_shift,
            "y_shift": y_shift,
            "shift_magnitude": float(np.hypot(x_shift, y_shift)),
        })
    return records


def _read_metadata_json(series_dir: Path) -> dict:
    metadata_path = series_dir / "metadata.json"
    if not metadata_path.exists():
        return {}
    with open(metadata_path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_registration_metadata(series_dir: Path, registration_payload: dict) -> Path:
    metadata_path = series_dir / "metadata.json"
    metadata = _read_metadata_json(series_dir)
    metadata["elastix_registration"] = registration_payload
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False, default=str)
    return metadata_path


def save_registered_tiff_stacks(
    stacks_dir: Path,
    registered_images: dict[str, sitk.Image],
    source_tiff_paths: dict[str, Path],
    output_dir_name: str = REGISTERED_STACKS_DIR_NAME,
) -> dict[str, Path]:
    output_dir = stacks_dir.parent / output_dir_name
    paths = {}
    for label, image in registered_images.items():
        source_path = source_tiff_paths[label]
        registered_arr = cast_array_like_tiff_dtype(sitk_image_to_array(image), source_path)
        output_path = output_dir / f"{source_path.stem}_registered{source_path.suffix}"
        paths[label] = write_tiff_stack(output_path, registered_arr)
    return paths

## 8. Elastix Registration Core

Reference channel에서 z-slice별 XY translation을 추정하고 같은 transform을 DAPI/Reference/Target 전체에 적용합니다.

In [ ]:
def make_identity_2d_transform(reference_slice_raw: sitk.Image) -> sitk.TranslationTransform:
    transform = sitk.TranslationTransform(2)
    transform.SetOffset((0.0, 0.0))
    return transform


def make_translation_2d_transform(tx: float, ty: float) -> sitk.TranslationTransform:
    transform = sitk.TranslationTransform(2)
    transform.SetOffset((float(tx), float(ty)))
    return transform


def apply_transform_to_slice(moving_slice_raw: sitk.Image, fixed_slice_raw: sitk.Image, transform: sitk.Transform) -> sitk.Image:
    registered_slice = sitk.Resample(
        moving_slice_raw,
        fixed_slice_raw,
        transform,
        sitk.sitkLinear,
        0.0,
        sitk.sitkFloat32,
    )
    return sitk.Cast(registered_slice, sitk.sitkFloat32)


def estimate_elastix_translation_transform(
    fixed_reference_slice_raw: sitk.Image,
    moving_reference_slice_raw: sitk.Image,
    metric_name: str,
    downsample_factor: int = REGISTRATION_XY_DOWNSAMPLE_FACTOR,
    max_iterations: int = ELASTIX_MAX_ITERATIONS,
    initial_offset: tuple[float, float] | None = None,
) -> tuple[sitk.TranslationTransform, list[float]]:
    """Estimate moving->fixed 2D translation with optional previous-shift recovery.

    When ``initial_offset`` is provided, the moving slice is first resampled
    with that previous offset, Elastix estimates only the residual shift, and
    the returned full-resolution transform is ``initial_offset + residual``.
    This gives the jump-recovery branch a real initialization mechanism while
    keeping output transforms in the same [0, x_shift, y_shift] schema.
    """
    if not HAS_ITK:
        raise ImportError("itk / itk-elastix가 설치되어 있지 않습니다. 위 설치 셀을 먼저 실행하세요.")
    import itk

    fixed_slice = preprocess_image(fixed_reference_slice_raw)
    moving_slice = preprocess_image(moving_reference_slice_raw)
    fixed_for_registration = downsample_image_for_registration(fixed_slice, factor=downsample_factor)
    moving_for_registration = downsample_image_for_registration(moving_slice, factor=downsample_factor)

    init_x, init_y = (0.0, 0.0) if initial_offset is None else (float(initial_offset[0]), float(initial_offset[1]))
    if initial_offset is not None and (abs(init_x) > 1e-12 or abs(init_y) > 1e-12):
        initial_transform = make_translation_2d_transform(init_x, init_y)
        moving_for_registration = apply_transform_to_slice(
            moving_for_registration,
            fixed_for_registration,
            initial_transform,
        )

    fixed_arr = sitk.GetArrayFromImage(fixed_for_registration).astype(np.float32)
    moving_arr = sitk.GetArrayFromImage(moving_for_registration).astype(np.float32)

    fixed_itk = itk.image_view_from_array(fixed_arr)
    moving_itk = itk.image_view_from_array(moving_arr)
    fixed_itk.SetSpacing(fixed_for_registration.GetSpacing())
    moving_itk.SetSpacing(moving_for_registration.GetSpacing())
    fixed_itk.SetOrigin(fixed_for_registration.GetOrigin())
    moving_itk.SetOrigin(moving_for_registration.GetOrigin())

    parameter_object = itk.ParameterObject.New()
    translation_map = parameter_object.GetDefaultParameterMap("translation")
    translation_map["Metric"] = [metric_name]
    translation_map["NumberOfResolutions"] = ["3"]
    translation_map["MaximumNumberOfIterations"] = [str(max_iterations)]
    translation_map["AutomaticTransformInitialization"] = ["false" if initial_offset is not None else "true"]
    if initial_offset is None:
        translation_map["AutomaticTransformInitializationMethod"] = ["GeometricalCenter"]
    parameter_object.AddParameterMap(translation_map)

    _, result_transform = itk.elastix_registration_method(
        fixed_itk,
        moving_itk,
        parameter_object=parameter_object,
        log_to_console=False,
    )

    transform_map = result_transform.GetParameterMap(0)
    residual_parameters = [float(v) for v in transform_map.get("TransformParameters", [])]
    if len(residual_parameters) < 2:
        raise ValueError(f"Unexpected Elastix transform parameters: {residual_parameters}")

    x_shift = init_x + residual_parameters[0]
    y_shift = init_y + residual_parameters[1]
    full_res_transform = make_translation_2d_transform(x_shift, y_shift)
    return full_res_transform, [0.0, x_shift, y_shift]


def run_elastix_reference_guided_translation(
    channel_raw_images: dict[str, sitk.Image],
    reference_label: str = REFERENCE_LABEL,
    metric_name: str = ELASTIX_METRIC,
    downsample_factor: int = REGISTRATION_XY_DOWNSAMPLE_FACTOR,
    max_iterations: int = ELASTIX_MAX_ITERATIONS,
    jump_threshold_voxels: float = JUMP_THRESHOLD_VOXELS,
    fatal_jump_threshold_voxels: float = FATAL_JUMP_THRESHOLD_VOXELS,
):
    channel_arrays = {label: sitk_image_to_array(image) for label, image in channel_raw_images.items()}
    reference_arr = channel_arrays[reference_label]
    shapes = {label: arr.shape for label, arr in channel_arrays.items()}
    if len(set(shapes.values())) != 1:
        raise ValueError(f"Shape mismatch across channels: {shapes}")

    registered_slices = {label: [] for label in channel_arrays}
    slice_transforms = []
    transform_log = []
    stop_conditions = []
    fallback_slices = []
    recovered_slices = []
    clipped_at = None

    for z in range(reference_arr.shape[0]):
        reference_slice_arr = reference_arr[z]
        reference_slice_raw = make_2d_sitk_image(reference_slice_arr, channel_raw_images[reference_label], z)

        if z == 0:
            for label, arr in channel_arrays.items():
                registered_slices[label].append(arr[z].astype(np.float32))
            slice_transforms.append([0.0, 0.0, 0.0])
            transform_log.append({"z": 0, "tx": 0.0, "ty": 0.0, "reason": "anchor"})
            stop_conditions.append("z=0: identity seed")
            continue

        fixed_previous_reference_slice = sitk.GetImageFromArray(registered_slices[reference_label][z - 1].astype(np.float32))
        fixed_previous_reference_slice.CopyInformation(reference_slice_raw)
        log_entry = {"z": int(z)}

        if not slice_has_signal(registered_slices[reference_label][z - 1], reference_slice_arr):
            transform = make_identity_2d_transform(fixed_previous_reference_slice)
            transform_parameters = [0.0, 0.0, 0.0]
            reason = "fallback_identity_insufficient_signal"
            fallback_slices.append(z)
        else:
            try:
                transform, transform_parameters = estimate_elastix_translation_transform(
                    fixed_previous_reference_slice,
                    reference_slice_raw,
                    metric_name=metric_name,
                    downsample_factor=downsample_factor,
                    max_iterations=max_iterations,
                    initial_offset=None,
                )
                reason = "ok"
            except Exception as exc:
                transform = make_identity_2d_transform(fixed_previous_reference_slice)
                transform_parameters = [0.0, 0.0, 0.0]
                reason = f"fallback_identity_failed: {exc}"
                fallback_slices.append(z)

        tx = float(transform_parameters[1])
        ty = float(transform_parameters[2])
        jump = None
        if z > 1 and slice_transforms:
            prev_tx = float(slice_transforms[-1][1])
            prev_ty = float(slice_transforms[-1][2])
            jump = float(np.hypot(tx - prev_tx, ty - prev_ty))
            log_entry["initial_jump"] = jump

            if jump > jump_threshold_voxels and reason == "ok":
                try:
                    recovered_transform, recovered_parameters = estimate_elastix_translation_transform(
                        fixed_previous_reference_slice,
                        reference_slice_raw,
                        metric_name=metric_name,
                        downsample_factor=downsample_factor,
                        max_iterations=max_iterations,
                        initial_offset=(prev_tx, prev_ty),
                    )
                    recovered_tx = float(recovered_parameters[1])
                    recovered_ty = float(recovered_parameters[2])
                    recovered_jump = float(np.hypot(recovered_tx - prev_tx, recovered_ty - prev_ty))
                    log_entry.update({
                        "recovery_initial_tx": prev_tx,
                        "recovery_initial_ty": prev_ty,
                        "recovered_tx": recovered_tx,
                        "recovered_ty": recovered_ty,
                        "recovered_jump": recovered_jump,
                    })
                    transform = recovered_transform
                    transform_parameters = recovered_parameters
                    tx, ty, jump = recovered_tx, recovered_ty, recovered_jump
                    recovered_slices.append(z)
                    reason = "ok_after_jump_recovery" if recovered_jump <= jump_threshold_voxels else "recovered_but_jump_remains_large"
                except Exception as exc:
                    tx, ty = prev_tx, prev_ty
                    jump = 0.0
                    transform = make_translation_2d_transform(tx, ty)
                    transform_parameters = [0.0, tx, ty]
                    reason = f"fallback_previous_transform_recovery_failed: {exc}"
                    fallback_slices.append(z)
                    log_entry["recovery_error"] = str(exc)

            if jump > fatal_jump_threshold_voxels:
                reason = "fatal_jump_circuit_breaker"
                log_entry.update({"tx": tx, "ty": ty, "reason": reason, "jump": jump})
                transform_log.append(log_entry)
                slice_transforms.append(transform_parameters)
                stop_conditions.append(f"z={z}: FATAL jump={jump:.2f}")
                clipped_at = z
                break

        for label, arr in channel_arrays.items():
            channel_slice_raw = make_2d_sitk_image(arr[z], channel_raw_images[label], z)
            registered_slice = apply_transform_to_slice(channel_slice_raw, fixed_previous_reference_slice, transform)
            registered_slices[label].append(sitk.GetArrayFromImage(registered_slice).astype(np.float32))
        slice_transforms.append(transform_parameters)
        log_entry.update({"tx": tx, "ty": ty, "reason": reason})
        if jump is not None:
            log_entry["jump"] = jump
        transform_log.append(log_entry)
        stop_conditions.append(f"z={z}: {reason}")

    registered_images = {
        label: compose_stack_from_slices(slices, channel_raw_images[label])
        for label, slices in registered_slices.items()
    }
    final_z = int(next(iter(registered_slices.values())).__len__())
    return {
        "registered_images": registered_images,
        "registered_reference": registered_images[reference_label],
        "registered_target": registered_images[TARGET_LABEL],
        "optimizer_metric": float("nan"),
        "stop_condition": " | ".join(stop_conditions[:8]) + (" | ..." if len(stop_conditions) > 8 else ""),
        "transform_parameters": slice_transforms,
        "transform_log": transform_log,
        "transform_summary": summarize_slice_transforms(slice_transforms),
        "n_slices": int(reference_arr.shape[0]),
        "final_z_slices": final_z,
        "n_successful_slices": final_z,
        "n_fallback_slices": len(fallback_slices),
        "fallback_slices": fallback_slices,
        "n_recovered_slices": len(recovered_slices),
        "recovered_slices": recovered_slices,
        "clipped_at": clipped_at,
        "downsample_factor": downsample_factor,
        "max_iterations": max_iterations,
        "jump_threshold_voxels": jump_threshold_voxels,
        "fatal_jump_threshold_voxels": fatal_jump_threshold_voxels,
    }

## 9. Discover Input Stack Sets

현재 `ROOT` 아래의 모든 `stacks` 폴더에서 DAPI/Reference/Target TIFF set을 찾습니다.

In [ ]:
stack_sets_df = discover_stack_sets(ROOT)
if stack_sets_df.empty:
    raise RuntimeError("처리할 DAPI/Reference/Target stacks 폴더를 찾지 못했습니다.")

display(stack_sets_df)
print(f"discovered stack sets: {len(stack_sets_df)}")

## 10. Run Registration

선택한 sample에 대해 iterative reference-guided registration을 실행하고, Reference에서 추정한 transform을 DAPI/Reference/Target 전체에 적용합니다.

In [ ]:
SELECT_SAMPLE_IDS = None
MAX_TEST_SLICES = None  # 전체 z-stack을 실행하려면 None으로 두세요.
USE_QUICK_TEST_MODE = QUICK_TEST_MODE
ACTIVE_DOWNSAMPLE_FACTOR = REGISTRATION_XY_DOWNSAMPLE_FACTOR if USE_QUICK_TEST_MODE else 1
ACTIVE_ELASTIX_MAX_ITERATIONS = ELASTIX_MAX_ITERATIONS if USE_QUICK_TEST_MODE else 256

selected_df = stack_sets_df.copy()
if SELECT_SAMPLE_IDS is not None:
    selected_df = selected_df[selected_df["sample_id"].isin(SELECT_SAMPLE_IDS)].reset_index(drop=True)
if selected_df.empty:
    raise RuntimeError(f"선택한 sample_id를 찾지 못했습니다: {SELECT_SAMPLE_IDS}")
print(f"selected stack sets: {len(selected_df)}")
display(selected_df)

results = []
registration_visuals = {}
METHOD_NAME = "Elastix iterative reference-guided translation-only"

for row in selected_df.itertuples(index=False):
    sample_id = row.sample_id
    stacks_dir = Path(row.stacks_dir)
    series_dir = Path(row.series_dir)
    source_tiff_paths = {
        label: Path(getattr(row, f"{label.lower()}_path"))
        for label in CHANNEL_LABELS
    }
    channel_raw_images = load_channel_images(row, channel_labels=CHANNEL_LABELS)
    if MAX_TEST_SLICES is not None:
        channel_raw_images = {
            label: image[:, :, :MAX_TEST_SLICES]
            for label, image in channel_raw_images.items()
        }

    reference_raw = channel_raw_images[REFERENCE_LABEL]
    target_raw = channel_raw_images[TARGET_LABEL]
    reference_proc = preprocess_image(reference_raw)
    target_proc = preprocess_image(target_raw)

    try:
        elastix_result = run_elastix_reference_guided_translation(
            channel_raw_images,
            reference_label=REFERENCE_LABEL,
            metric_name=ELASTIX_METRIC,
            downsample_factor=ACTIVE_DOWNSAMPLE_FACTOR,
            max_iterations=ACTIVE_ELASTIX_MAX_ITERATIONS,
            jump_threshold_voxels=JUMP_THRESHOLD_VOXELS,
            fatal_jump_threshold_voxels=FATAL_JUMP_THRESHOLD_VOXELS,
        )
        registered_images = elastix_result["registered_images"]
        registered_reference = registered_images[REFERENCE_LABEL]
        registered_target = registered_images[TARGET_LABEL]
        registered_reference_proc = preprocess_image(registered_reference)
        registered_target_proc = preprocess_image(registered_target)

        registered_stacks_dir = series_dir / REGISTERED_STACKS_DIR_NAME
        saved_tiffs = {label: None for label in CHANNEL_LABELS}
        if SAVE_REGISTERED_TIFF_STACKS:
            saved_tiffs = save_registered_tiff_stacks(
                stacks_dir=stacks_dir,
                registered_images=registered_images,
                source_tiff_paths=source_tiff_paths,
            )
            print("saved registered TIFF stacks:")
            for label, path in saved_tiffs.items():
                print(f"  {label:<10s} -> {path}")

        before_metrics = evaluate_adjacent_stack_metrics(reference_proc)
        after_metrics = evaluate_adjacent_stack_metrics(registered_reference_proc)
        delta_metrics = metric_delta(before_metrics, after_metrics)
        transform_records = build_transform_provenance_records(
            transform_parameters=elastix_result["transform_parameters"],
            sample_id=sample_id,
            method=METHOD_NAME,
            registered_stacks_dir=registered_stacks_dir,
        )

        registration_payload = {
            "method": METHOD_NAME,
            "registration_metric": ELASTIX_METRIC,
            "reference_channel": REFERENCE_LABEL,
            "applied_channels": list(CHANNEL_LABELS),
            "fixed_image": "registered Reference slice at z-1",
            "moving_image": "raw Reference slice at z",
            "transform": "translation (2D)",
            "transform_parameters_schema": ["reserved", "x_shift", "y_shift"],
            "transform_parameters": elastix_result["transform_parameters"],
            "transform_records_csv_schema": [
                "sample_id",
                "method",
                "registered_stacks_dir",
                "z",
                "x_shift",
                "y_shift",
                "shift_magnitude",
            ],
            "transform_records": transform_records,
            "transform_summary": elastix_result["transform_summary"],
            "downsample_factor": elastix_result["downsample_factor"],
            "max_iterations": elastix_result["max_iterations"],
            "jump_threshold_voxels": elastix_result["jump_threshold_voxels"],
            "fatal_jump_threshold_voxels": elastix_result["fatal_jump_threshold_voxels"],
            "n_slices": elastix_result["n_slices"],
            "final_z_slices": elastix_result["final_z_slices"],
            "n_fallback_slices": elastix_result["n_fallback_slices"],
            "fallback_slices": elastix_result["fallback_slices"],
            "n_recovered_slices": elastix_result["n_recovered_slices"],
            "recovered_slices": elastix_result["recovered_slices"],
            "clipped_at": elastix_result["clipped_at"],
            "transform_log": elastix_result["transform_log"],
            "quality_metrics": {
                "evaluated_channel": REFERENCE_LABEL,
                "before": before_metrics,
                "after": after_metrics,
                "delta": delta_metrics,
            },
            "csv_outputs": {
                "metrics_summary_enabled": SAVE_METRICS_SUMMARY_CSV,
                "metrics_summary_path": str(METRICS_OUTPUT_CSV),
                "transform_parameters_enabled": SAVE_TRANSFORM_PARAMETERS_CSV,
                "transform_parameters_path": str(registered_stacks_dir / TRANSFORM_PARAMETERS_CSV_NAME),
            },
            "saved_tiffs": {label: str(path) if path else "" for label, path in saved_tiffs.items()},
        }
        metadata_path = None
        if SAVE_METADATA_JSON:
            metadata_path = save_registration_metadata(series_dir, registration_payload)
            print(f"saved metadata: {metadata_path}")

        results.append({
            "sample_id": sample_id,
            "method": METHOD_NAME,
            "registration_metric": ELASTIX_METRIC,
            "optimizer_metric": elastix_result["optimizer_metric"],
            "stop_condition": elastix_result["stop_condition"],
            "transform_parameters": json.dumps(elastix_result["transform_parameters"]),
            "transform_summary": elastix_result["transform_summary"],
            "n_slices": elastix_result["n_slices"],
            "final_z_slices": elastix_result["final_z_slices"],
            "n_successful_slices": elastix_result["n_successful_slices"],
            "n_fallback_slices": elastix_result["n_fallback_slices"],
            "n_recovered_slices": elastix_result["n_recovered_slices"],
            "clipped_at": elastix_result["clipped_at"],
            "downsample_factor": elastix_result["downsample_factor"],
            "max_iterations": elastix_result["max_iterations"],
            "stacks_dir": str(stacks_dir),
            "registered_stacks_dir": str(registered_stacks_dir),
            "metadata_path": str(metadata_path) if metadata_path else "",
            **{f"registered_{label.lower()}_tiff": str(saved_tiffs[label]) if saved_tiffs[label] else "" for label in CHANNEL_LABELS},
            **{f"before_{k}": v for k, v in before_metrics.items()},
            **{f"after_{k}": v for k, v in after_metrics.items()},
            **{f"delta_{k}": v for k, v in delta_metrics.items()},
        })

        registration_visuals[(sample_id, METHOD_NAME)] = {
            "fixed_image": reference_proc,
            "moving_image": reference_proc,
            "registered_image": registered_reference_proc,
            "raw_target": target_raw,
            "preprocessed_target": target_proc,
            "registered_target": registered_target,
            "registered_target_preprocessed": registered_target_proc,
            "raw_reference": reference_raw,
            "preprocessed_reference": reference_proc,
            "registered_reference": registered_reference,
            "registered_reference_preprocessed": registered_reference_proc,
            "raw_dapi": channel_raw_images["DAPI"],
            "registered_dapi": registered_images["DAPI"],
            "registered_stacks_dir": str(registered_stacks_dir),
        }

    except Exception as exc:
        print(f"Elastix failed for {sample_id}: {exc}")
        results.append({
            "sample_id": sample_id,
            "method": METHOD_NAME,
            "registration_metric": ELASTIX_METRIC,
            "optimizer_metric": np.nan,
            "stop_condition": f"FAILED: {exc}",
            "transform_parameters": "[]",
        })

results_df = pd.DataFrame(results)
display(results_df)

## 11. Metrics Summary

registration 전후 metric 변화량을 sample별 요약 표로 정리하고, 옵션에 따라 CSV로 저장합니다.


In [ ]:
# 필요 시 결과 표만 따로 정리해서 보세요.
# metric은 Target_stack이 아니라 adjacent Reference_stack slice 간 보정 전후 유사도를 평가합니다.
if not results_df.empty:
    cols = [
        "sample_id",
        "registration_metric",
        "after_mean_squares",
        "after_mutual_information",
        "after_normalized_mutual_information",
        "after_pearson_r",
        "delta_mean_squares",
        "delta_mutual_information",
        "delta_normalized_mutual_information",
        "delta_pearson_r",
        "n_slices",
        "final_z_slices",
        "n_successful_slices",
        "n_fallback_slices",
        "n_recovered_slices",
        "clipped_at",
        "downsample_factor",
        "max_iterations",
        "stacks_dir",
        "registered_stacks_dir",
        "metadata_path",
        "registered_dapi_tiff",
        "registered_reference_tiff",
        "registered_target_tiff",
    ]
    summary_df = results_df[[c for c in cols if c in results_df.columns]].sort_values(["sample_id"])
    display(summary_df)
    if SAVE_METRICS_SUMMARY_CSV:
        summary_df.to_csv(METRICS_OUTPUT_CSV, index=False)
        print(f"saved metrics csv: {METRICS_OUTPUT_CSV}")
else:
    print("results_df가 비어 있습니다.")

## 12. Transform Plot Configuration

표시할 sample/method 필터를 설정합니다.


In [ ]:
# Slice별 registration transform을 x/y shift로 확인합니다.
# transform_parameters는 호환성을 위해 [0.0, x_translation, y_translation] 순서로 저장됩니다.
TRANSFORM_PLOT_SAMPLE_IDS = None  # 예: ["1_PSD Ms MA1046_Homer Rb SYSY/DIW_63x_FiX ITS_1"]
TRANSFORM_PLOT_METHODS = None  # 예: ["Elastix iterative reference-guided translation-only"]


## 13. Transform Table and Plot Helpers

JSON으로 저장된 slice transform을 표로 펼치고 x/y shift plot을 그리는 함수입니다.


In [ ]:
def make_transform_parameters_df(results_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if results_df.empty or "transform_parameters" not in results_df.columns:
        return pd.DataFrame(rows)

    for result in results_df.itertuples(index=False):
        params_json = getattr(result, "transform_parameters", "[]")
        try:
            params = json.loads(params_json) if isinstance(params_json, str) else params_json
        except json.JSONDecodeError:
            params = []

        rows.extend(build_transform_provenance_records(
            transform_parameters=params,
            sample_id=result.sample_id,
            method=result.method,
            registered_stacks_dir=Path(getattr(result, "registered_stacks_dir", "")),
        ))
    return pd.DataFrame(rows)


def plot_registration_transforms(
    transform_df: pd.DataFrame,
    sample_ids: list[str] | None = None,
    methods: list[str] | None = None,
    save_plots: bool = SAVE_TRANSFORM_SHIFT_PLOTS,
) -> None:
    if transform_df.empty:
        print("표시할 transform parameter가 없습니다. 먼저 registration 셀을 실행하세요.")
        return

    plot_df = transform_df.copy()
    if sample_ids is not None:
        plot_df = plot_df[plot_df["sample_id"].isin(sample_ids)]
    if methods is not None:
        plot_df = plot_df[plot_df["method"].isin(methods)]
    if plot_df.empty:
        print("필터 조건에 맞는 transform parameter가 없습니다.")
        return

    for (sample_id, method), group in plot_df.groupby(["sample_id", "method"], sort=False):
        group = group.sort_values("z")
        fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
        fig.suptitle(f"{sample_id} | {method}")

        axes[0].plot(group["z"], group["x_shift"], marker="o", label="x translation")
        axes[0].plot(group["z"], group["y_shift"], marker="o", label="y translation")
        axes[0].axhline(0, color="0.3", linewidth=0.8)
        axes[0].set_ylabel("translation (pixel/physical unit)")
        axes[0].legend()
        axes[0].grid(alpha=0.3)

        axes[1].plot(group["z"], group["shift_magnitude"], marker="o", color="tab:green")
        axes[1].set_ylabel("shift magnitude")
        axes[1].grid(alpha=0.3)

        axes[1].set_xlabel("z slice")

        plt.tight_layout()
        if save_plots:
            output_dir = Path(group["registered_stacks_dir"].iloc[0])
            output_dir.mkdir(parents=True, exist_ok=True)
            output_path = output_dir / TRANSFORM_SHIFT_PLOT_NAME
            fig.savefig(output_path, dpi=200, bbox_inches="tight")
            print(f"saved transform shift plot: {output_path}")
        plt.show()

## 14. Transform Diagnostics

transform table을 만들고, 옵션에 따라 CSV와 shift plot을 저장합니다.


In [ ]:
transform_parameters_df = make_transform_parameters_df(results_df)
display(transform_parameters_df)
if SAVE_TRANSFORM_PARAMETERS_CSV and not transform_parameters_df.empty:
    for output_dir, group in transform_parameters_df.groupby("registered_stacks_dir", sort=False):
        output_path = Path(output_dir) / TRANSFORM_PARAMETERS_CSV_NAME
        output_path.parent.mkdir(parents=True, exist_ok=True)
        group.to_csv(output_path, index=False)
        print(f"saved transform parameters csv: {output_path}")
plot_registration_transforms(
    transform_parameters_df,
    sample_ids=TRANSFORM_PLOT_SAMPLE_IDS,
    methods=TRANSFORM_PLOT_METHODS,
    save_plots=SAVE_TRANSFORM_SHIFT_PLOTS,
)

## 15. Before/After Overlay GIFs

Reference는 초록색, Target은 빨간색으로 표시해 registration 전후 overlay GIF를 비교합니다.


In [ ]:
# Registration 전후 overlay GIF 두 개만 notebook 안에서 비교합니다.
# Reference_stack은 초록색, Target_stack은 빨간색으로 표시됩니다.
GIF_SAMPLE_IDS = None  # 예: ["1_PSD Ms MA1046_Homer Rb SYSY/DIW_63x_FiX ITS_1"]
GIF_METHODS = None  # 예: ["Elastix iterative reference-guided translation-only"]
GIF_DURATION_MS = 120
GIF_DISPLAY_WIDTH = 320

items = list(registration_visuals.items())
if GIF_SAMPLE_IDS is not None:
    items = [(key, value) for key, value in items if key[0] in GIF_SAMPLE_IDS]
if GIF_METHODS is not None:
    items = [(key, value) for key, value in items if key[1] in GIF_METHODS]

if not items:
    print("표시할 registration GIF 결과가 없습니다. 먼저 registration 셀을 실행하거나 필터를 확인하세요.")
else:
    for (sample_id, method), payload in items:
        show_before_after_overlay_gif_comparison(
            sample_id=sample_id,
            method=method,
            reference_raw_image=payload["raw_reference"],
            target_raw_image=payload["raw_target"],
            registered_reference_image=payload["registered_reference"],
            registered_target_image=payload["registered_target"],
            duration_ms=GIF_DURATION_MS,
            display_width=GIF_DISPLAY_WIDTH,
        )
        if SAVE_OVERLAY_GIFS:
            output_dir = Path(payload["registered_stacks_dir"])
            before_path = output_dir / BEFORE_OVERLAY_GIF_NAME
            after_path = output_dir / AFTER_OVERLAY_GIF_NAME
            save_overlay_gif(payload["raw_reference"], payload["raw_target"], before_path, duration_ms=GIF_DURATION_MS)
            save_overlay_gif(payload["registered_reference"], payload["registered_target"], after_path, duration_ms=GIF_DURATION_MS)
            print(f"saved overlay GIFs: {before_path} | {after_path}")